In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency
import seaborn as sns
from sklearn.cluster import AgglomerativeClustering, AffinityPropagation, KMeans
from sklearn.mixture import GaussianMixture
from statsmodels.stats.multitest import multipletests
from sklearn.decomposition import PCA
from sklearn.metrics import pairwise_distances

In [28]:
#Chi-squared for cluster membership vs. surgery before/after
group_annotations = pd.read_csv('../data/processed/group_annotation.csv', index_col=0)
top_variable_expression=pd.read_csv('../results/top_5000_variable_expression.csv', index_col=0)
group_annotations = group_annotations.loc[top_variable_expression.T.index]

k=3
data_for_clustering = top_variable_expression.T

agglo = AgglomerativeClustering(n_clusters=k, compute_distances=True)
agglo_cluster_labels = agglo.fit_predict(data_for_clustering)

kmeans = KMeans(n_clusters=k, random_state=27)
kmeans_labels = kmeans.fit_predict(data_for_clustering)

gmm = GaussianMixture(n_components=k, random_state=42, covariance_type='full')
gmm_labels = gmm.fit_predict(data_for_clustering)

X_pca = PCA(n_components=2, random_state=42).fit_transform(data_for_clustering)
similarity = -pairwise_distances(X_pca, metric='euclidean')
ap = AffinityPropagation(affinity='euclidean', random_state=3)
ap_labels = ap.fit_predict(similarity)



In [29]:
def chi_squared_clusters_vs_surgery(cluster_labels):
   contingency_table = pd.crosstab(cluster_labels, group_annotations['Group']=='T0')
   chi2, p, _, _ = chi2_contingency(contingency_table)
   #res = pd.DataFrame({'Genes 1': n1, 'Genes 2': n2, 'Chi2 Statistic': chi2, 'p-value': p})
  
   return chi2, p

In [32]:
chi2, p = chi_squared_clusters_vs_surgery(agglo_cluster_labels)
print(f"Agglomerative: chi2={chi2:.2f}, p-value={p:.6f}")

chi2, p = chi_squared_clusters_vs_surgery(kmeans_labels)
print(f"KMeans: chi2={chi2:.2f}, p-value={p:.6f}")

chi2, p = chi_squared_clusters_vs_surgery(gmm_labels)
print(f"Gaussian mixture models: chi2={chi2:.2f}, p-value={p:.6f}")

chi2, p = chi_squared_clusters_vs_surgery(ap_labels)
print(f"Affinity propagation: chi2={chi2:.2f}, p-value={p:.6f}")

Agglomerative: chi2=17.50, p-value=0.000159
KMeans: chi2=24.41, p-value=0.000005
Gaussian mixture models: chi2=12.91, p-value=0.001576
Affinity propagation: chi2=29.46, p-value=0.000542


Adjustment for Multiple Hypothesis Testing

In [34]:

# Example: run for multiple clustering results
results = []
clusterings = {
    'Agglomerative': agglo_cluster_labels,
    'KMeans': kmeans_labels,
    'Gaussian Mixture Models': gmm_labels,
    'Affinity Propagation': ap_labels
}

for name, labels in clusterings.items():
    chi2, p = chi_squared_clusters_vs_surgery(labels)
    results.append({'Method': name, 'Chi2': chi2, 'p-value': p})

results_df = pd.DataFrame(results)

# Adjust p-values for multiple testing (Benjamini-Hochberg FDR)
results_df['adj_p-value'] = multipletests(results_df['p-value'], method='fdr_bh')[1]

print(results_df)

                    Method       Chi2   p-value  adj_p-value
0            Agglomerative  17.497277  0.000159     0.000317
1                   KMeans  24.405280  0.000005     0.000020
2  Gaussian Mixture Models  12.906177  0.001576     0.001576
3     Affinity Propagation  29.459064  0.000542     0.000723
